In [0]:
import json
from datetime import datetime

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_metadata", "")

run_id = dbutils.widgets.get("run_id")
table_metadata_str = dbutils.widgets.get("table_metadata")

if not run_id or not table_metadata_str:
    raise ValueError("run_id and table_metadata are required")

table_metadata = json.loads(table_metadata_str.replace("'", '"'))
table_id = table_metadata["table_id"]
table_name = table_metadata["table_name"]

start_time = datetime.utcnow()

print("Run ID:", run_id)
print("Table ID:", table_id)
print("Table Name:", table_name)

Run ID: 1234567890
Table ID: 7
Table Name: customer_360


/home/spark-a8cdd856-86e4-45e9-928e-19/.ipykernel/2060/command-5132122931787984-664439943:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()


In [0]:
# spark.sql("CREATE SCHEMA IF NOT EXISTS banking.gold")

# COMMAND ----------

entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs 
        SET
            layer = 'Gold',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Gold',
            TIMESTAMP('{start_time}'),
            NULL,
            'INPROGRESS',
            NULL,
            NULL
        )
    """)

print("Audit entry created / updated")





Audit entry created / updated


In [0]:
# COMMAND ----------

notebook_path = f"gold_transformations/{table_name}"
print("Notebook to execute:", notebook_path)


Notebook to execute: gold_transformations/customer_360


In [0]:
# COMMAND ----------

status = "SUCCESS"
error_message = None
records = None

try:
    result = dbutils.notebook.run(
        notebook_path,
        timeout_seconds=0
    )
    if result:
        records = int(result)
    print("Notebook completed successfully")
    print("Records:", records)

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    print("Notebook failed")
    print(error_message)

Notebook completed successfully
Records: 4000


In [0]:

# COMMAND ----------

end_time = datetime.utcnow()

spark.sql(f"""
    UPDATE banking.metadata.pipeline_runs
    SET
        end_time = TIMESTAMP('{end_time}'),
        status = '{status}',
        number_of_records = {records if records else 'NULL'},
        error_message = {f"'{error_message}'" if error_message else 'NULL'}
    WHERE run_id = {run_id}
    AND table_id = {table_id}
""")

print("Audit table updated")



/home/spark-32d9bff0-28e6-4286-8f68-88/.ipykernel/2053/command-5132122931787989-3226199373:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


Audit table updated


In [0]:
# COMMAND ----------

if status == "FAILED":
    raise Exception(error_message)